# Analisis dan Pemodelan Konsumsi Energi Gedung (Kasus 02)

Notebook ini mengikuti brief dosen. Target adalah konsumsi listrik per jam (`konsumsi_kWh`). Prediksi dibuat sehari sebelumnya, sehingga seluruh fitur konsumsi historis harus memiliki jeda minimal 24 jam. Baseline wajib adalah nilai pada jam yang sama tujuh hari sebelumnya (`lag_168`).

**Penting:** data test Mei-Juni 2026 tidak digunakan pada EDA atau pemilihan model. Test hanya dievaluasi setelah model dipilih dan dikunci; ringkasan hasil akhirnya tersedia pada bagian terakhir notebook.


## 1. Persiapan pustaka dan lokasi proyek

In [ ]:
from pathlib import Path
import sys
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display, Image

ROOT = Path.cwd()
if not (ROOT / 'data' / 'raw.csv').exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from src.features import clean_hourly_data, cleaning_summary, prepare_xy, FEATURE_COLUMNS
from src.train import load_before_test, train_validate_and_fit

DATA_PATH = ROOT / 'data' / 'raw.csv'
ARTIFACT_DIR = ROOT / 'artifacts'
FIGURE_DIR = ROOT / 'reports' / 'figures'
print('Folder proyek:', ROOT)
print('Data tersedia:', DATA_PATH.exists())


## 2. Memahami data dan memeriksa kualitasnya

Pada tahap ini hanya bagian data sebelum 1 Mei 2026 yang dibaca. Dengan demikian, test masih ditahan.

In [ ]:
raw_pretest = load_before_test(DATA_PATH)
print('Jumlah baris sebelum test:', len(raw_pretest))
display(raw_pretest.head())

quality = cleaning_summary(raw_pretest)
display(pd.DataFrame([quality]).T.rename(columns={0: 'Jumlah'}))


### Pembersihan data

Timestamp duplikat digabungkan dengan median per waktu. Indeks dibentuk per jam agar lag dihitung berdasarkan waktu, bukan sekadar baris. Konsumsi negatif dan konsumsi di atas 1.000 kWh diperlakukan sebagai nilai tidak valid. Jam yang tidak memiliki rekaman tetap kosong agar tidak membuat data konsumsi palsu.

Maret 2026 tidak tersedia, sehingga validation akan menggunakan data April 2026 yang ada. Kekurangan ini dicatat sebagai keterbatasan dataset.


In [ ]:
hourly = clean_hourly_data(raw_pretest)
print('Rentang waktu:', hourly.index.min(), 'sampai', hourly.index.max())
print('Jumlah jam pada indeks:', len(hourly))
print('Jumlah target valid:', hourly['konsumsi_kWh'].notna().sum())


## 3. Exploratory Data Analysis (EDA)

In [ ]:
valid_target = hourly['konsumsi_kWh'].dropna()
eda_summary = {
    'jumlah_target_valid': int(valid_target.size),
    'rata_rata_kWh': round(float(valid_target.mean()), 2),
    'median_kWh': round(float(valid_target.median()), 2),
    'minimum_kWh': round(float(valid_target.min()), 2),
    'maksimum_kWh': round(float(valid_target.max()), 2),
}
display(pd.DataFrame([eda_summary]))

by_hour = valid_target.groupby(valid_target.index.hour).mean()
fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(by_hour.index, by_hour.values, marker='o')
ax.set_title('Rata-rata konsumsi berdasarkan jam')
ax.set_xlabel('Jam dalam sehari')
ax.set_ylabel('Rata-rata konsumsi (kWh)')
ax.set_xticks(range(0, 24, 2))
ax.grid(alpha=0.25)
plt.show()


In [ ]:
paired = hourly[['okupansi_orang', 'konsumsi_kWh']].dropna()
if len(paired) > 3000:
    paired = paired.sample(3000, random_state=42)
fig, ax = plt.subplots(figsize=(8, 4))
ax.scatter(paired['okupansi_orang'], paired['konsumsi_kWh'], alpha=0.3, s=12)
ax.set_title('Hubungan okupansi dengan konsumsi listrik')
ax.set_xlabel('Jumlah orang')
ax.set_ylabel('Konsumsi (kWh)')
plt.show()


## 4. Membuat fitur tanpa kebocoran data masa depan

Fitur konsumsi yang digunakan adalah lag 24, 48, 168, dan 336 jam. Fitur rata-rata rolling juga dihitung dari data yang sudah digeser 24 jam. Kelembapan tidak dipakai karena brief hanya meminta prakiraan suhu dan okupansi untuk masukan aplikasi.

In [ ]:
X, y = prepare_xy(hourly)
print('Jumlah fitur:', len(FEATURE_COLUMNS))
print('Daftar fitur:')
print(FEATURE_COLUMNS)
display(X.head(10))


## 5. Pembagian waktu dan perbandingan model

Train sampai 28 Februari 2026. Validation menggunakan data tersedia pada Maret-April; karena Maret kosong, sampel validasi berasal dari April. Sampel dibandingkan pada baris yang memiliki target valid dan `lag_168` yang tersedia, sehingga baseline dan model diuji pada baris yang sama.

In [ ]:
train_start, validation_start, test_start = (
    pd.Timestamp('2026-01-01'), pd.Timestamp('2026-03-01'), pd.Timestamp('2026-05-01')
)
train_mask = (hourly.index < validation_start) & y.notna() & X['lag_168'].notna()
validation_mask = (hourly.index >= validation_start) & (hourly.index < test_start) & y.notna() & X['lag_168'].notna()
print('Sampel train:', int(train_mask.sum()))
print('Sampel validation:', int(validation_mask.sum()))

# Jalankan pipeline pemilihan model. Fungsi ini hanya membaca data sampai 30 April 2026.
training_summary = train_validate_and_fit(DATA_PATH)
validation_table = pd.read_csv(ARTIFACT_DIR / 'validation_metrics.csv')
display(validation_table.sort_values('MAE_kWh'))
print('Model final:', training_summary['model_final'])


## 6. Fitur yang berpengaruh dan artefak model

Random Forest dipilih karena memberikan MAE terendah pada validation. Feature importance adalah ukuran kontribusi relatif di dalam model dan tidak boleh ditafsirkan sebagai bukti sebab-akibat.


In [ ]:
importance_path = ARTIFACT_DIR / 'feature_importance.csv'
if importance_path.exists():
    importance = pd.read_csv(importance_path)
    display(importance.head(12))
    display(Image(filename=str(FIGURE_DIR / '04_feature_importance.png')))

metadata = json.loads((ARTIFACT_DIR / 'metadata.json').read_text(encoding='utf-8'))
print('Versi model:', metadata['model_version'])
print('Nama model:', metadata['model_name'])
print('Lag minimum:', metadata['minimum_target_lag_hours'], 'jam')
print('Model tersimpan:', (ARTIFACT_DIR / 'model.joblib').exists())


## 7. Hasil test akhir (sudah dievaluasi setelah model dikunci)

Hasil di bawah dibaca dari file ringkasan evaluasi yang dibuat setelah pemilihan model. Notebook ini **tidak membuka ulang data test** dan hasilnya tidak digunakan untuk tuning.


In [ ]:
test_metrics_path = ARTIFACT_DIR / 'test_metrics.json'
if test_metrics_path.exists():
    test_metrics = json.loads(test_metrics_path.read_text(encoding='utf-8'))
    display(pd.DataFrame([
        {'Model': 'Baseline - lag 168', **test_metrics['baseline']},
        {'Model': metadata['model_name'], **test_metrics['model']},
    ]))
    print('Jumlah sampel test:', test_metrics['test_rows'])
    print('Periode:', test_metrics['test_period_start'], 'sampai', test_metrics['test_period_end'])
else:
    print('Hasil test belum tersedia. Jalankan evaluasi akhir hanya setelah model dipilih dan dikunci.')


## 8. Kesimpulan singkat

Model Random Forest mengungguli baseline pada validation dan test. Fitur okupansi menjadi fitur dengan importance paling besar pada data ini, kemudian konsumsi pada jam yang sama minggu lalu dan suhu luar. Model masih merupakan prototipe akademik karena data bersifat sintetis dan memiliki kekosongan periode.

**Peringatan:** jangan mengubah model berdasarkan skor test. Jika model diubah, diperlukan test set baru untuk evaluasi yang tidak bias.
